# Salish Sea Model (SSM) Metabolic Index Analysis - all node test -Salmon at 6.1kg
**Date:** 2025.07.18  
**Purpose:** Production implementation of Tim Essington's linear regression metabolic index model for Salish Sea salmon habitat analysis


## Notebook Overview

This production notebook implements metabolic index calculations for Chinook salmon using Tim Essington's updated linear regression methodology. The metabolic index quantifies how well organisms can meet metabolic demands given environmental oxygen availability and physiological constraints.

### Key Features:
- **Tim Essington's calc_mi() function**: Linear regression model with 95% confidence intervals
- **Vectorized processing**: Optimized for large SSM datasets (16,012 nodes × 361 days × 10 depths)
- **Memory-optimized workflow**: Sequential processing and export to manage memory usage
- **Multi-species ready**: Easy species switching via parameter blocks
- **Input mapping system**: Flexible configuration for different data sources (min/max/avg)
- **Quality assurance**: Built-in validation against Tim's original test cases

### Workflow Steps:
1. **Data Loading**: Load SSM saturation outputs and reference datasets
2. **Subsampling**: Optional node selection for development/testing
3. **Species Configuration**: Select species parameters (salmon/crab/sculpin/sole)
4. **Metabolic Calculations**: Process routine and basal metabolism with confidence intervals
5. **Export**: Save results to NetCDF files with compression
6. **Quality Assurance**: Validate against reference test cases

### Inputs:
- **SSM Saturation Outputs**: Environmental data from prior saturation analysis
  - **Source directory**: `../../../../SSM_output/SSM_saturation/`
  - **Data types**: 3D arrays (time × depth × nodes) containing e.g::
    - `CalMinParam_3D_pO2_daily_min_kPa/` - Minimum partial pressure of oxygen (kPa)
    - `CalMinParam_3D_temp_daily_mean_CT/` - Maximum daily temperature (°C)
    - `CalMaxParam_3D_pO2_daily_max_kPa/` - Maximum partial pressure of oxygen (kPa)
    - `CalMaxParam_3D_temp_daily_mean_CT/` - Maximum daily temperature (°C)
  - **Datasets within each**: `exist.nc` (existing conditions), `wqm_reference.nc` (reference conditions)
  - **Dimensions**: 361 days × 10 depth layers × 16,012 nodes (subsampled to 3 nodes for testing)

- **Species Parameters**: Configured directly in notebook
  - **Model coefficients**: `betas` array (4 elements) from fitted regression
  - **Uncertainty matrix**: `var_covar` (4×4) for confidence intervals
  - **Organism weight**: Species-specific body weight in grams
  - **Taxa identifier**: Used for output file naming

- **Helper Functions**: Python utilities loaded from working directory
  - `helper_ExportsAndFigs.py` - Export and compression functions
  - `helper_variable_name_datasetreview.py` - Data subsampling and QA functions

### Outputs:
- **Metabolic Index Arrays**: 3D xarray datasets with same structure as inputs
  - Routine metabolism (active swimming) MI values
  - Basal/SMR metabolism (resting) MI values  
  - 95% confidence intervals (upper/lower bounds) for both metabolism types
- **Export Format**: NetCDF files with compression to `../../../../SSM_output/SSM_metabolic/`
- **Naming Convention**: `{CalMinParam|CalMaxParam}_3D_{taxa}_Mindex_{routine|basal}[_ci_{upper|lower}]`
- **Quality Assurance**: Validation outputs against Tim Essington's reference test cases

### Workflow production details 
**run multiple species scripts simultaneously on hyak**
Code designed for species independently processed from same sources:
  - Output files include {taxa} variable in names: CalMinParam_3D_{taxa}_Mindex_routine
  - Dictionary keys include species: SSMcalcs_dic['CalMinParam_3D_salmon_Mindex_routine']
  - Export directories are species-specific: SSM_metabolic/CalMinParam_3D_salmon_Mindex_routine/

Steps to run multiple species:
  1. Copy script to separate directories (e.g., salmon/, crab/)
  2. Edit species parameter block in each copy (uncomment different species)
  3. Submit separate SLURM jobs pointing to same input data
  4. All outputs export to same SSM_metabolic/ folder without conflicts

  5. To run parallel test with same species data:
  - Change only the taxa variable. e.g: taxa = "salmon_test1" (line in species parameter block)
  - Keep all other parameters identical (betas, var_covar, organism_weight_grams)
  - Outputs become: CalMinParam_3D_salmon_test1_Mindex_routine/
  - No other code changes needed - taxa variable controls all file naming

  

**PROCESSING TIME ESTIMATE BASED ON 3-NODE TEST**
  Hyak Test Results: 3 nodes processed in 38.21 seconds total
  - First calculation block: 3.88 seconds
  - First export block: 3.75 seconds
  - Data points per calculation: 10,830 (3 nodes × 361
  days × 10 depths)

  Hyak estimate based on actual 16,012 NODE run:
  - Routine metabolism calculation: 20,652.36
  seconds (~5.7 hours)
  - Export operations: ~few seconds each
  - Total expected runtime: ~6+ hours for all
  calculations

  **Production Estimates:**
  - Routine metabolism: ~5.7 hours
  - SMR metabolism: ~5.7 hours (similar
  complexity)
  - All confidence intervals: ~4 × 5.7 hours =
  ~23 hours
  - Total full pipeline: ~34+ hours for all
  outputs

## Core Implementation:  Essington et al. (in prep) Metabolic Index Function
This section contains the essential metabolic index calculation functions developed for Essington et al (in prep). These functions implement a linear regression model with confidence intervals for calculating metabolic index values including partial pressure of oxygen (pO2) and temperature which will be taken from the Salish Sea Model outputs.

In [1]:
# Library Imports and Configuration
import numpy as np  
from scipy.stats import norm  
import sys

sys.stdout.flush() # Force Python to flush output immediately (so SLURM .out file updates in real time)


In [2]:
# Utility Function - Temperature Conversion
def kelvin(temperature_c):
    temperature_K = temperature_c + 273.15  
    return temperature_K

In [3]:
# Main Metabolic Index Calculation Function
def calc_mi(pO2, w, temperature, betas, var_covar, method="smr", confidence_level=0.95):
    """
    Calculate the metabolic index (MI) and its confidence interval.

    Parameters:
    - pO2: partial pressure of O2
    - w: body size (g)
    - temperature: degrees Celsius
    - betas: parameter estimates array length 4 (fitted model coefficients??)
    - var_covar: 4x4 variance-covariance matrix (parameter uncertainty)
    - method: 'smr' or other (determines x_predict structure) eg 'smr' (standard metabolic rate) or 'routine' (active metabolism)
    - confidence_level: e.g., 0.95 for 95% CI   
   
    Returns:
    - dict with keys: mi, lower_bound, upper_bound
    """
    
    # Define in function reference values and physical constants (that don't change between organisms or method eg standard/routine metabolism)
    wref = 5  # Reference body weight in grams for scaling 
    tref = 15  # Reference temperature in Celsius for thermal scaling
    kb = 8.617333262145E-5  # Boltzmann constant in eV/K for temperature effects
    # From constants, calculate scaled predictors for allometric and thermal relationships
    # modify logw and inv_temperature:
    logw = np.log(w / wref)  # Log-transform body size ratio for allometric scaling
    inv_temperature = (1 / kb) * (1 / kelvin(temperature) - 1 / kelvin(tref))  # Arrhenius temperature scaling
    
    # Construct predictor vector (1d array with 4 elements), where we do different things on the last element depending:
    # on whether wish mi based on SMR or on routine metabolism.  If neither, return an error
    # Vector format: [intercept, body_size, temperature, metabolic_mode: smr/routine]
    if method == "smr":  # Standard metabolic rate (resting metabolism)
        x_predict = np.array([-1.0, logw, inv_temperature, -1.0])  # SMR flag = -1
    elif method == "routine":  # Routine metabolic rate (active metabolism)
        x_predict = np.array([-1.0, logw, inv_temperature, 0.0])  # Routine flag = 0
    else:
        raise ValueError("Invalid method. 'method' should be either 'smr' or 'routine'.")
    
    # Calculate predicted log(MI) using pre-trained model coefficients via matrix algebra
    log_mi_predict = np.dot(x_predict, betas) + np.log(pO2)  # Matrix multiplication for linear combination plus oxygen effect - Where dot product: sum of element-wise products
    
    # Calculate standard error of log(MI)- Calculate prediction uncertainty using error propagation
    var_pred = np.dot(x_predict, np.dot(var_covar, x_predict))  # Quadratic form: x^T * Σ * x for prediction variance
    log_mi_se = np.sqrt(var_pred)  # Convert variance to standard error
    
    # Calculate confidence interval bounds on log scale
    #     Quantile for two-tailed confidence interval
    z_score = norm.ppf(0.5 + confidence_level / 2.0)  # Critical value from standard normal distribution
    #     Confidence interval on log scale
    log_lower_bound = log_mi_predict - z_score * log_mi_se  # Lower bound = mean - critical_value * std_error
    log_upper_bound = log_mi_predict + z_score * log_mi_se  # Upper bound = mean + critical_value * std_error
    
    # Return exponentiated values: maximum likelihood, lower and upper bound of CI 
    # Transformed back to original scale using exponential function
    return {
        "mi": np.exp(log_mi_predict),  # Exponentiate to get maximum ?? likelihood ?? metabolic index value
        "lower_bound": np.exp(log_lower_bound),  # Exponentiate lower CI bound
        "upper_bound": np.exp(log_upper_bound)   # Exponentiate upper CI bound
    }

## DEBUG: Tests 1 to 4 validating core metabolic function - can be removed in production code 

In [4]:
# Test code conditions inputs- Chinook Salmon
w = 5  # Body weight in grams (reference size)
temperature = 15  # Temperature in Celsius (reference temperature)
method = "routine"  # Metabolic state: routine (active) vs SMR (resting)

# Model coefficients from fitted regression (Chinook Salmon specific)
betas = np.array([1.58422927, -0.04328307, 0.17567401, -0.32428962])  # [intercept, size_effect, temp_effect, metabolic_mode]

# Variance-covariance matrix of parameter estimates (captures parameter uncertainty)
var_covar = np.array([
    [0.173846857, 1.326809e-02, -0.073963952, 6.287643e-03],  # Variances and covariances for intercept
    [0.013268092, 8.014129e-03, -0.004246992, 1.119302e-05],  # Variances and covariances for size effect
    [-0.073963952, -4.246992e-03, 0.035758738, -2.752385e-03],  # Variances and covariances for temperature effect
    [0.006287643, 1.119302e-05, -0.002752385, 2.331340e-02]   # Variances and covariances for metabolic mode
])

BR: this one makes sense. If you specify p = p_crit, you should get a MI of 1.0. And p_crit is going to be something you can pull out of the model coefficients.

In [5]:
# Test Case 1 - Baseline Routine Metabolism (Should = 1.0)
# Set pO2 to critical value where MI = 1 (pO2 = V and method = "routine"
pO2 = np.exp(betas[0])  # Use exp of intercept coefficient to set critical oxygen level
result = calc_mi(pO2, w, temperature, betas, var_covar, method="routine", confidence_level=0.95)  # Calculate MI with uncertainty
print(f"pO2 used where = V: {pO2}, with  results as follows (MI should be =1.0):")  # Should be approximately exp(1.58422927)
print(f"Baseline routine MI: {result['mi']}")  # Should be approximately 1.0
print(f"Lower bound: {result['lower_bound']}")  # Lower confidence limit
print(f"Upper bound: {result['upper_bound']}")  # Upper confidence limit

pO2 used where = V: 4.875532211106204, with  results as follows (MI should be =1.0):
Baseline routine MI: 1.0
Lower bound: 0.4416639693994982
Upper bound: 2.2641647706957735


BR: also makes sense. The active metabolism is higher than standard rate, so if computing at standard rate the MI would be higher.

In [6]:
# Test Case 2 - Standard Metabolic Rate Comparison (Should > 1.0)
# Same conditions but using SMR instead of routine metabolism
result = calc_mi(pO2, w, temperature, betas, var_covar, method="smr", confidence_level=0.95)  # SMR has lower metabolic demands
print(f"Standard MI and same pO2 ({pO2}), with  results as follows (should be >1 as requires less O2):")  # Higher MI because SMR requires less oxygen than routine
print(f"Standard MI: {result['mi']}")  # Higher MI because SMR requires less oxygen than routine
print(f"Lower bound: {result['lower_bound']}")  # Lower confidence limit
print(f"Upper bound: {result['upper_bound']}")  # Upper confidence limit

Standard MI and same pO2 (4.875532211106204), with  results as follows (should be >1 as requires less O2):
Standard MI: 1.3830478074262176
Lower bound: 0.5636549567839213
Upper bound: 3.3936031513686356


BR: makes sense, although it looks like this should be compared with test case 1 and the output is a little ambigous there, it still passes.

In [7]:
# Test Case 3 - Temperature Effect (Increasing (higher) Temperature should decrease MI)
temperature = 20  # Increase temperature by 5°C from reference
result = calc_mi(pO2, w, temperature, betas, var_covar, method="routine", confidence_level=0.95)  # Higher temperature increases metabolic demand
print(f"Standard MI ,same p02, and higher temp ({temperature}c), with  results as follows:")  
print(f"Higher temperature MI (should be lower than prior): {result['mi']}")  # Lower MI due to increased metabolic rate at higher temperature
print(f"Lower bound: {result['lower_bound']}")  # Lower confidence limit  
print(f"Upper bound: {result['upper_bound']}")  # Upper confidence limit

Standard MI ,same p02, and higher temp (20c), with  results as follows:
Higher temperature MI (should be lower than prior): 0.8863271883375078
Lower bound: 0.49374434953562796
Upper bound: 1.5910579746889557


BR: makes sense here too. Comparing against Test Case 1.

In [8]:
# Test Case 4 - Body Size Effect (Larger organisms should have lower MI)
w = 4000  # Increase body weight to 4kg (800x larger than reference)
temperature = 15  # Reset temperature to reference value
result = calc_mi(pO2, w, temperature, betas, var_covar, method="routine", confidence_level=0.95)  # Larger fish have higher mass-specific metabolic demands
print(f"Standard MI, same pO2, and larger body size MI (should be smaller MI than prior) with results as follows")  # Lower MI due to allometric scaling effects
print(f"Larger body size MI (should be <1): {result['mi']}")  # Lower MI due to allometric scaling effects
print(f"Lower bound: {result['lower_bound']}")  # Lower confidence limit
print(f"Upper bound: {result['upper_bound']}")  # Upper confidence limit

Standard MI, same pO2, and larger body size MI (should be smaller MI than prior) with results as follows
Larger body size MI (should be <1): 0.7487646847944787
Lower bound: 0.2330737289969287
Upper bound: 2.40545579979442


## Vectorization for SSM Data Processing

Converted functions to handle large numpy arrays efficiently for Salish Sea Model (SSM) data processing. This section creates vectorized versions of the metabolic index calculations that can process entire 3D datasets (time × depth × nodes) simultaneously.

*BR review: there is a lot of optimization potential here. A simple improvement would be to replace the sets of three different vectorized routines with one. Change calc_mi to return a small array of `[mi, lower_bound, upper_bound]` rather than a dict, and make a single call to vectorize with argument `otypes=[float, float, float]` so it knows to expect three floats as output. Then when calling the vectorized version with three N-length arrays, you'll get a 3xN array returned with all the data in N invocations of calc_mi rather than 3N invocations; it should be three times faster. SMR invocations would still need to be separate.*

*To improve further would require modifications to the calc_mi function to matrix math internally, and from a cursory look that appears to be possible.*

BR: confirmed vectorization test passed, but it only tests the first cell. Let's make the test a little more thorough. Tests passed, all results match.

In [25]:
# Vectorization is needed. Simplest approach with minimal code change: Create separate vectorized functions for each output
# This avoids the complexity of handling dictionary returns and mixed array types
# CONSOLIDATED: Create ALL vectorized functions at once (routine + SMR) for better organization

# ROUTINE METABOLISM: Vectorized functions for active/swimming metabolism
vectorized_mi_routine = np.vectorize(lambda pO2, w, temp: calc_mi(pO2, w, temp, betas, var_covar, method="routine")['mi'])  # Extract MI value only
vectorized_lower_routine = np.vectorize(lambda pO2, w, temp: calc_mi(pO2, w, temp, betas, var_covar, method="routine")['lower_bound'])  # Extract lower CI bound
vectorized_upper_routine = np.vectorize(lambda pO2, w, temp: calc_mi(pO2, w, temp, betas, var_covar, method="routine")['upper_bound'])  # Extract upper CI bound

# SMR METABOLISM: Vectorized functions for standard/resting metabolism
vectorized_mi_smr = np.vectorize(lambda pO2, w, temp: calc_mi(pO2, w, temp, betas, var_covar, method="smr")['mi'])  # Extract MI value only
vectorized_lower_smr = np.vectorize(lambda pO2, w, temp: calc_mi(pO2, w, temp, betas, var_covar, method="smr")['lower_bound'])  # Extract lower CI bound
vectorized_upper_smr = np.vectorize(lambda pO2, w, temp: calc_mi(pO2, w, temp, betas, var_covar, method="smr")['upper_bound'])  # Extract upper CI bound

print("✓ Created all vectorized functions: routine + SMR metabolism")

# Test with single values first
print("\nTesting vectorized functions with single values...")

# Test with arrays (this is where vectorization provides benefit)
pO2_array = np.array([21.0, 15.0, 10.0])  # Multiple oxygen levels
w_array = np.array([5, 100, 1000])        # Multiple body weights  
temp_array = np.array([15, 18, 22])       # Multiple temperatures

for i,(test_pO2,test_w,test_temp) in enumerate(zip(pO2_array,w_array,temp_array)):
    print(f"\nSingle value test #{i+1}")
    test_mi = vectorized_mi_routine(test_pO2, test_w, test_temp)         # Get routine MI value
    test_lower = vectorized_lower_routine(test_pO2, test_w, test_temp)   # Get routine lower bound
    test_upper = vectorized_upper_routine(test_pO2, test_w, test_temp)   # Get routine upper bound

    print(f"Single value test MI: {test_mi}")       # Should match original function
    print(f"Lower bound: {test_lower}")             # Should match original function
    print(f"Upper bound: {test_upper}")             # Should match original function

print("\nTesting with arrays...")

# Apply vectorized functions to arrays:
mi_results = vectorized_mi_routine(pO2_array, w_array, temp_array)         # Returns array of MI values
lower_results = vectorized_lower_routine(pO2_array, w_array, temp_array)   # Returns array of lower bounds
upper_results = vectorized_upper_routine(pO2_array, w_array, temp_array)   # Returns array of upper bounds

print(f"Array MI results: {mi_results}")       # Array of metabolic index values
print(f"Array lower bounds: {lower_results}")  # Array of lower confidence bounds  
print(f"Array upper bounds: {upper_results}")  # Array of upper confidence bounds
print("✓ All vectorized functions created and tested!")

✓ Created all vectorized functions: routine + SMR metabolism

Testing vectorized functions with single values...

Single value test #1
Single value test MI: 4.307222081758194
Lower bound: 1.9023448017144942
Upper bound: 9.752260497079815

Single value test #2
Single value test MI: 2.512448127759285
Lower bound: 1.2684303628618065
Upper bound: 4.976540911902517

Single value test #3
Single value test MI: 1.3788314706977778
Lower bound: 0.5834581776942732
Upper bound: 3.2584618697088445

Testing with arrays...
Array MI results: [4.30722208 2.51244813 1.37883147]
Array lower bounds: [1.9023448  1.26843036 0.58345818]
Array upper bounds: [9.7522605  4.97654091 3.25846187]
✓ All vectorized functions created and tested!


## Production Pipeline: SSM Data Loading and Processing

This section implements the production pipeline for processing Salish Sea Model data through the metabolic index calculations. The workflow includes:

1. **Data Loading**: Load SSM saturation outputs and reference datasets using `load_all_nc_datasets()`
2. **Subsampling**: Optional node selection for development/testing using `subsample_ssm_data()`
3. **QA Override**: Development mode for exact validation against Tim's test cases
4. **Species Configuration**: Select and configure species-specific parameters
5. **Memory-Optimized Processing**: Sequential calculation and export of all metabolic indices
6. **Quality Assurance**: Validate results against expected patterns and test cases

### Key Features:
- **Memory optimization**: Process one output type at a time to manage large datasets
- **Input mapping system**: Flexible configuration for different data sources via `SSMinputsForMetabolic`
- **Multi-species ready**: Easy switching between salmon/crab/sculpin/sole parameters
- **Export compatibility**: Results saved in same structure as previous workflow

### Data Loading from SSM Saturation Outputs

Load processed saturation analysis results and reference SSM datasets. The `load_all_nc_datasets()` function efficiently loads multiple NetCDF files from specified subdirectories, organizing them into nested dictionaries for processing.

In [10]:
## Data from saturation outputs were in which are loaded directly using load_all_nc_datasets* 
#output_dir = '../../../../SSM_output/SSM_saturation' 
#* Note: the input directory is defined below and passed in call to load_all_nc_datasets pointing to the prior scripts "output_dir" directory (e.g. saturation outputs)

#initialize
from helper_ExportsAndFigs               import export_dictionary_of_nc_datasets

In [11]:
#load specific folders in directory -define below 

import xarray as xr  # Import xarray for handling NetCDF files
import os  # Import os for directory operations


def load_all_nc_datasets(output_dir, subdirectories_to_load):
    """
    Load all NetCDF datasets from specified subdirectories within a  given directory.

    Parameters:
    output_dir (str): Path to the output directory containing subdirectories with NetCDF files.
    subdirectories_to_load (list): A list of subdirectory names to load datasets from.

    Returns:
    dict: A dictionary of dictionaries containing xarray Datasets.
          The outer dictionary keys are the subdirectory names,
          and the inner dictionary keys are the NetCDF file names (without extension).
    """
    datasets = {}  # Initialize an empty dictionary to store the datasets

    # Iterate over each specified subdirectory
    for subdirectory in subdirectories_to_load:
        subdirectory_path = os.path.join(output_dir, subdirectory)  # Get the full path of the subdirectory
        if os.path.isdir(subdirectory_path):  # Check if it is a directory
            datasets[subdirectory] = {}  # Initialize a dictionary for the subdirectory

            # Iterate over each NetCDF file in the subdirectory
            for nc_file in os.listdir(subdirectory_path):
                if nc_file.endswith('.nc'):  # Check if the file is a NetCDF file
                    file_path = os.path.join(subdirectory_path, nc_file)  # Get the full path of the NetCDF file
                    dataset_name = os.path.splitext(nc_file)[0]  # Get the file name without extension
                    datasets[subdirectory][dataset_name] = xr.open_dataset(file_path)  # Load the dataset
        else:
            print(f"Directory not found: {subdirectory_path}")

    return datasets  # Return the dictionary of dictionaries containing the datasets
#######################Call:

#load SSM2014_dic datasets: 
output_dir = '../../../../SSM_output/SSM_data_working' # Define the output directory where current data is
# Specify the subdirectories to load
subdirectories_to_load = [  'Calculated_WholeYear10Layers_3D_Xarray']
# Call the function to load all NetCDF datasets from the specified subdirectories
loaded_datasets = load_all_nc_datasets(output_dir, subdirectories_to_load)
# Make  dictionary using result of call to function above
SSM2014_dic = loaded_datasets
del loaded_datasets, subdirectories_to_load # Cleanup

#load SSMcalcs_dic datasets: 
output_dir = '../../../../SSM_output/SSM_saturation'  # Define the input directory of prior script outputs to load 

# Specify the subdirectories to load
subdirectories_to_load = [  'CalMinParam_3D_pO2_daily_min_kPa',    
                            'CalMinParam_3D_temp_daily_mean_CT',
                            'CalMaxParam_3D_pO2_daily_max_kPa',
                            'CalMaxParam_3D_temp_daily_mean_CT'                            
                            ] 
# Call the function to load all NetCDF datasets from the specified subdirectories
loaded_datasets = load_all_nc_datasets(output_dir, subdirectories_to_load)
# Make dictionary using result of call to function above
SSMcalcs_dic = loaded_datasets
del loaded_datasets, subdirectories_to_load # Cleanup


print(f"\nNow separated into individual dictionaries and the loaded_datasets dictionary deleted")
print(f"Minimal error checking so if you get an error on variables not found to del etc, likely inputs are not in the SSM_output folder as expected\n\n")


Now separated into individual dictionaries and the loaded_datasets dictionary deleted
Minimal error checking so if you get an error on variables not found to del etc, likely inputs are not in the SSM_output folder as expected




In [12]:
#reset main output directory to use in this processing 
output_dir = '../../../../SSM_output/'


### Subsampling for Development and Testing

For development and QA testing, I have a subsample function selecting from the full SSM dataset (16,012 nodes) to a smaller set of representative nodes. This allows faster testing and validation while maintaining the same data structure and processing workflow. The `subsample_ssm_data()` function preserves all dataset relationships while reducing computational requirements.
NOTE: must set exact subdirectories to sub-sample

BR uncommented next two cells per instructions. Learned that Ctrl+/ toggles comments over multiple selected lines.

In [15]:
# =============================================================================
# DEBUG/Dev: subsampling function to SSM data -only for arrays in specified subdirectories_to_subsample:
# =============================================================================
#Import functions from helper file for subsampling SSM data
from helper_variable_name_datasetreview import subsample_ssm_data


# Define parameters for subsampling
specified_nodes = [13789, 8841, 14409]  # Specific node IDs for testing (1-based indexing)
# Note node number matches GIS (1-based indexing) will be -1 to (convert to 0-based indexing)for actuall xarray (*.nc) operations
subdirectories_to_subsample = [
    'CalMinParam_3D_pO2_daily_min_kPa',    
    'CalMinParam_3D_temp_daily_mean_CT',
    'CalMaxParam_3D_pO2_daily_max_kPa',
    'CalMaxParam_3D_temp_daily_mean_CT'                            
] #note these should be set to exactly what is loaded earlier unless for specfic case

# Apply subsampling function with in-place modification (most memory efficient)
print("APPLYING SUBSAMPLING FUNCTION TO SSM DATA")
print("="*60)

SSMcalcs_dic = subsample_ssm_data(SSMcalcs_dic, subdirectories_to_subsample, specified_nodes)

print("\n" + "="*60)
print("NOTE: CHECK No OF NODES ABOVE IS AS DEFINED IN SCRIPT - SUBSAMPLING FUNCTION COMPLETE")
print("="*60)
print("SSMcalcs_dic now contains subsampled data")
print("All following code will work with the subsampled data")
print("To run full dataset: Comment out the function call above")
print("="*60)

APPLYING SUBSAMPLING FUNCTION TO SSM DATA
SUBSAMPLING SSM DATA TO SPECIFIC NODES
Subsampling to 3 nodes: [13789, 8841, 14409]
(Using 0-based indices: [13788, 8840, 14408])

Processing subdirectory: CalMinParam_3D_pO2_daily_min_kPa
  Processing dataset: wqm_reference
    pO2_daily_min_kPa: (361, 10, 16012) → (361, 10, 3)
  Completed dataset: wqm_reference
  Processing dataset: exist
    pO2_daily_min_kPa: (361, 10, 16012) → (361, 10, 3)
  Completed dataset: exist

Processing subdirectory: CalMinParam_3D_temp_daily_mean_CT
  Processing dataset: exist
    temp_daily_mean_CT: (361, 10, 16012) → (361, 10, 3)
  Completed dataset: exist
  Processing dataset: wqm_reference
    temp_daily_mean_CT: (361, 10, 16012) → (361, 10, 3)
  Completed dataset: wqm_reference

Processing subdirectory: CalMaxParam_3D_pO2_daily_max_kPa
  Processing dataset: wqm_reference
    pO2_daily_max_kPa: (361, 10, 16012) → (361, 10, 3)
  Completed dataset: wqm_reference
  Processing dataset: exist
    pO2_daily_max_kPa:

### QA Override for Exact Validation (Development Mode)

**FOR PRODUCTION: Comment out the QA override block in the next cell and comment in production block following.**

This QA override system allows exact validation of the processing pipeline by setting all input data to Tim Essington's Test 1 reference values (pO2=4.876 kPa, temp=15°C, weight=5g). When these exact conditions are used, the metabolic index calculations should produce the known expected results, validating that the entire processing pipeline is working correctly.

The `override_ssm_data_for_qa()` function systematically replaces all pO2 and temperature values in the SSM datasets with the test reference values, enabling exact comparison with Tim's original function results.

In [16]:
# =============================================================================
# QA OVERRIDE BLOCK - COMMENTED OUT FOR PRODUCTION
# =============================================================================

# Import QA override function from helper file
from helper_variable_name_datasetreview import override_ssm_data_for_qa

# Apply QA overrides using the flexible function from helper
test_pO2_value = np.exp(betas[0])  # Calculate Test 1 pO2 value from first beta coefficient (4.875532211106204)
test_temp_value = 15.0             # Test 1 temperature value in Celsius
organism_weight_grams = 5.0        # Override production weight (4104.7g) with Test 1 weight (5g)

print(f"QA MODE: Using Test 1 values for exact validation")  # f-string formatting for variable insertion
print(f"  Weight: {organism_weight_grams}g (Test 1 reference)")  # Show current weight being used

# Call the flexible QA override function with parameters from above
SSMcalcs_dic = override_ssm_data_for_qa(
    ssm_dictionary=SSMcalcs_dic,  # Pass the main SSM data dictionary
    subdirectories_to_override=subdirectories_to_subsample,  # Use same subdirectory list as subsampling function (defined earlier)
    pO2_override_value=test_pO2_value,  # Override all pO2 with Test 1 value
    temp_override_value=test_temp_value  # Override all temperature with Test 1 value
)

print(f"\nExpected result: All MI calculations should equal 1.0")  # Explain what should happen with Test 1 values
print(f"FOR PRODUCTION: Comment out the entire QA OVERRIDE BLOCK above")  # Instruction for production use



QA MODE: Using Test 1 values for exact validation
  Weight: 5.0g (Test 1 reference)
QA OVERRIDE: Setting custom values in SSM data for validation

Processing subdirectory: CalMinParam_3D_pO2_daily_min_kPa
  Processing dataset: wqm_reference
  Processing dataset: exist

Processing subdirectory: CalMinParam_3D_temp_daily_mean_CT
  Processing dataset: exist
  Processing dataset: wqm_reference

Processing subdirectory: CalMaxParam_3D_pO2_daily_max_kPa
  Processing dataset: wqm_reference
  Processing dataset: exist

Processing subdirectory: CalMaxParam_3D_temp_daily_mean_CT
  Processing dataset: wqm_reference
  Processing dataset: exist

QA Override complete - 0 data arrays modified
All pO2 data set to: 4.876 kPa
All temperature data set to: 15.0°C

Expected result: All MI calculations should equal 1.0
FOR PRODUCTION: Comment out the entire QA OVERRIDE BLOCK above


In [17]:
# #OR comment out this block instead to run QA
# # =============================================================================
# # METABOLIC INDEX CALCULATIONS ON FULL SSM DATA - UNCOMMENT FOR PRODUCTION VERSION
# # =============================================================================

# print("="*80)
# print("PROCESSING FULL SSM DATA WITH VECTORIZED METABOLIC FUNCTIONS")
# print("="*80)

# print("PRODUCTION MODE: Using actual SSM environmental data")
# print(f"✓ QA override is commented out")
# print(f"✓ Processing with real pO2 and temperature values")
# # =============================================================================

### Species-Specific Parameter Configuration

Configure species-specific physiological parameters for metabolic index calculations. Parameters include:

- **betas**: Fitted regression coefficients for the linear metabolic model
- **var_covar**: Variance-covariance matrix capturing parameter uncertainty for confidence intervals  
- **organism_weight_grams**: Representative body weight for the species
- **taxa**: Species identifier for output file naming

**Instructions**: Only ONE species block should be active at a time. Comment/uncomment the desired species section below. Current options include Chinook salmon, Dungeness crab, Staghorn sculpin, and English sole.

BR: for usability it might be good to put these values in a YAML file, then one just has to specify which species key to consider.

In [18]:
# =============================================================================
# SPECIES-SPECIFIC INPUT PARAMETERS
# =============================================================================
# Instructions: Comment/uncomment the desired species block below
# Only ONE species block should be active at a time
# Each block defines: betas, var_covar, organism_weight_grams, taxa

# =============================================================================
# SALMON (Chinook) - ACTIVE
# =============================================================================
# Model coefficients from fitted regression (Chinook Salmon specific)
betas = np.array([1.58422927, -0.04328307, 0.17567401, -0.32428962])  # [log(V), n, Eo, beta_method] ie.[intercept, size_effect, temp_effect, metabolic_mode]
# Critical oxygen threshold at reference conditions (5g, 15°C, routine metabolism)
Chinook_V = np.exp(1.58422927)  # = 4.88 kPa (exponential converts log value to actual kPa)
# Variance-covariance matrix of parameter estimates (captures parameter uncertainty)
var_covar = np.array([
    [0.173846857, 1.326809e-02, -0.073963952, 6.287643e-03],  # Variances and covariances for intercept
    [0.013268092, 8.014129e-03, -0.004246992, 1.119302e-05],  # Variances and covariances for size effect
    [-0.073963952, -4.246992e-03, 0.035758738, -2.752385e-03],  # Variances and covariances for temperature effect
    [0.006287643, 1.119302e-05, -0.002752385, 2.331340e-02]   # Variances and covariances for metabolic mode
])
# Species-specific parameters
#organism_weight_grams = 4104.7  # Body weight (grams) - typical adult Chinook salmon  #old NOAA
organism_weight_grams = 6400 # Body weight (grams) - typical adult Chinook salmon #new  see papers cited in report
taxa = "salmon"  # Taxa name for file naming

# =============================================================================
# CRAB (Dungeness) - COMMENT OUT TO DISABLE
# =============================================================================
# # Model coefficients from fitted regression (Dungeness Crab specific)
# betas = np.array([1.39504370, -0.06864861, 0.29907692, -0.32428962])  # [log(V), n, Eo, beta_method] ie.[intercept, size_effect, temp_effect, metabolic_mode]
# # Critical oxygen threshold at reference conditions (5g, 15°C, routine metabolism)
# Crab_V = np.exp(1.39504370)  # = 4.04 kPa (exponential converts log value to actual kPa)
# # Variance-covariance matrix of parameter estimates (captures parameter uncertainty)
# var_covar = np.array([
#     [0.160201361, 0.006825386, -0.043946093, 0.005911341],   # Variances and covariances for intercept
#     [0.006825386, 0.006126607, -0.003405334, -0.000179495],  # Variances and covariances for size effect
#     [-0.043946093, -0.003405334, 0.026743298, -0.001186574], # Variances and covariances for temperature effect
#     [0.005911341, -0.000179495, -0.001186574, 0.023313400]   # Variances and covariances for metabolic mode
# ])
# # Species-specific parameters
# organism_weight_grams = 1200.0  # Body weight (grams) - typical adult Dungeness crab
# taxa = "crab"  # Taxa name for file naming

# =============================================================================
# SCULPIN (Staghorn) - COMMENT OUT TO DISABLE
# =============================================================================
# # Model coefficients from fitted regression (Staghorn Sculpin specific)
# betas = np.array([1.6134040, -0.1212538, 0.3281826, -0.3242896])  # [log(V), n, Eo, beta_method] ie.[intercept, size_effect, temp_effect, metabolic_mode]
# # Critical oxygen threshold at reference conditions (5g, 15°C, routine metabolism)
# Sculpin_V = np.exp(1.6134040)  # = 5.02 kPa (exponential converts log value to actual kPa)
# # Variance-covariance matrix of parameter estimates (captures parameter uncertainty)
# var_covar = np.array([
#     [0.037644692, 0.004643649, -0.011180171, -0.000406770],  # Variances and covariances for intercept
#     [0.004643649, 0.006018756, -0.003176410, -0.000159905],  # Variances and covariances for size effect
#     [-0.011180171, -0.003176410, 0.019672155, -0.000398393], # Variances and covariances for temperature effect
#     [-0.000406770, -0.000159905, -0.000398393, 0.023313400]  # Variances and covariances for metabolic mode
# ])
# # Species-specific parameters
# organism_weight_grams = 1000  # Body weight (grams) - USING MAX WEIGHT CURRENTLY UNKNOWN -REPLACE WITH REAL adult Staghorn sculpin
# taxa = "sculpin"  # Taxa name for file naming

# =============================================================================
# SOLE (English) - COMMENT OUT TO DISABLE
# =============================================================================
# # Model coefficients from fitted regression (English Sole specific)
# betas = np.array([1.21586046, -0.07752212, 0.26795349, -0.32428962])  # [log(V), n, Eo, beta_method] ie.[intercept, size_effect, temp_effect, metabolic_mode]
# # Critical oxygen threshold at reference conditions (5g, 15°C, routine metabolism)
# Sole_V = np.exp(1.21586046)  # = 3.37 kPa (exponential converts log value to actual kPa)
# # Variance-covariance matrix of parameter estimates (captures parameter uncertainty)
# var_covar = np.array([
#     [0.133984605, -0.000759200, -0.034233516, 0.002470956],  # Variances and covariances for intercept
#     [-0.000759200, 0.005736930, -0.001583776, -0.000338378], # Variances and covariances for size effect
#     [-0.034233516, -0.001583776, 0.025289992, -0.000539733], # Variances and covariances for temperature effect
#     [0.002470956, -0.000338378, -0.000539733, 0.023313400]   # Variances and covariances for metabolic mode
# ])
# # Species-specific parameters
# organism_weight_grams = 1000.0  # Body weight (grams) - good sized English sole
# taxa = "sole"  # Taxa name for file naming

print(f"✓ Species parameters loaded for: {taxa}")
print(f"  Weight: {organism_weight_grams}g")
print(f"  Critical V: {np.exp(betas[0]):.3f} kPa")
print(f"  Betas: {betas}")


✓ Species parameters loaded for: salmon
  Weight: 6400g
  Critical V: 4.876 kPa
  Betas: [ 1.58422927 -0.04328307  0.17567401 -0.32428962]


### Metabolic Index Call and Processing

**Core Production Pipeline**: This section implements the memory-optimized processing approach that handles large SSM datasets efficiently. 

**Key Features:**
- **Input Mapping System**: `SSMinputsForMetabolic` dictionary provides flexible configuration for different data sources
- **Sequential Processing**: Each output type (routine, basal, confidence intervals) is calculated and exported separately to manage memory
- **Immediate Export**: Results are exported immediately after calculation using the same compression and structure as previous workflows
- **Performance Timing**: Built-in timing measurements for performance assessment

**Outputs Generated:**
1. **Routine Metabolism**: Active swimming metabolic index 
2. **Basal Metabolism**: Standard/resting metabolic rate index
3. **Confidence Intervals**: 95% upper and lower bounds for both metabolism types

All results are saved to `../../../../SSM_output/SSM_metabolic/` with taxa-specific naming and NetCDF compression.

In [19]:
# =============================================================================
# APPLY METABOLIC INDEX CALCULATIONS TO SUBSAMPLED SSM DATA - MEMORY OPTIMIZED
# =============================================================================

# =============================================================================
#### UPFRONT MAPPING OF INPUT DATASETS AND VARIABLE NAMES
# =============================================================================
# This dictionary defines all input data sources for metabolic calculations
# Modify for different variable names, min/max/avg choices, or new datasets

SSMinputsForMetabolic = {
    'CalMinParam': {
        'pO2_subdir': 'CalMinParam_3D_pO2_daily_min_kPa',
        'pO2_var': 'pO2_daily_min_kPa',
        'temp_subdir': 'CalMinParam_3D_temp_daily_mean_CT',
        'temp_var': 'temp_daily_mean_CT',
    },
    'CalMaxParam': {
        'pO2_subdir': 'CalMaxParam_3D_pO2_daily_max_kPa',
        'pO2_var': 'pO2_daily_max_kPa',
        'temp_subdir': 'CalMaxParam_3D_temp_daily_mean_CT',
        'temp_var': 'temp_daily_mean_CT',
    },
}

print(f"\n Input mapping configured for metabolic calculations:")
for param_type, config in SSMinputsForMetabolic.items():
    print(f"  {param_type}: pO2 from {config['pO2_var']}, temp from {config['temp_var']}")

# =============================================================================
# APPLY METABOLIC INDEX CALCULATIONS TO SUBSAMPLED SSM DATA - MEMORY OPTIMIZED
# =============================================================================

# Import timing module for performance measurement
import time

# Setup export parameters 
output_dir_export_nc = f'{output_dir}/SSM_metabolic'
encoding = {'zlib': True, 'complevel': 4}  # Same compression as old version

# Start overall processing timer
overall_start_time = time.time()
routine_calc_time = 0  #default timing for routine calculations
routine_export_time = 0  #default timing for routine export

print("\nProcessing subsampled SSM data through vectorized metabolic functions")
print("Memory-optimized approach: Processing one output type at a time")
print(f"Exporting to: {output_dir_export_nc}")
print(f"Processing species: {taxa}")
print("-" * 60)

##################################################################
# ROUTINE Metabolic Calculation loop 
print("\n" + "="*50)
print("PROCESSING: Routine Metabolism")
print("="*50)

routine_start_time = time.time()# Start timing for first calculation block

for param_type, config in SSMinputsForMetabolic.items():
    print(f"\nProcessing {param_type}...")
    
    # Direct reference to mapped subdirectories and variables
    pO2_subdir = config['pO2_subdir']
    pO2_var = config['pO2_var']
    temp_subdir = config['temp_subdir']
    temp_var = config['temp_var']
    
    print(f"  Using pO2 from: {pO2_subdir}[{pO2_var}]")
    print(f"  Using temp from: {temp_subdir}[{temp_var}]")
    
    # Process each dataset (e.g., 'exist', 'wqm_reference')
    for key in SSMcalcs_dic[pO2_subdir].keys():
        print(f"  Processing dataset: {key}")
        
        # Direct variable extraction using mapped names
        pO2_data = SSMcalcs_dic[pO2_subdir][key][pO2_var]
        temp_data = SSMcalcs_dic[temp_subdir][key][temp_var]
        
        print(f"    Data shapes: pO2={pO2_data.shape}, temp={temp_data.shape}")
        
        # Flatten data for vectorized calculation
        pO2_flat = pO2_data.values.flatten()  # Convert xarray to numpy and flatten
        temp_flat = temp_data.values.flatten()  # Convert xarray to numpy and flatten
        
        # Use organism weight (either QA override or production value)
        weight_flat = np.full_like(pO2_flat, organism_weight_grams)  # Weight array same shape
        
        print(f"    Using organism weight: {organism_weight_grams}g")
        print(f"    Calculating routine metabolic indices for {len(pO2_flat):,} data points...")
        
        # Apply vectorized metabolic index calculations - 95% CI (confidence_level=0.95) - this is 95/5 not 90/10
        # Process routine metabolism only
        routine_mi_flat = vectorized_mi_routine(pO2_flat, weight_flat, temp_flat)
        
        # Create xarray results using simplified approach
        routine_mi_xarray = pO2_data.copy(data=routine_mi_flat.reshape(pO2_data.shape))
        routine_mi_xarray.name = f'Mindex_{taxa}_routine'
        
        # Create separate dictionary keys for each output (matches old workflow pattern)
        routine_key = f"{param_type}_3D_{taxa}_Mindex_routine"
        
       # Initialize dictionary using subsampled data template (matches current data dimensions by using routine key)
              # This ensures that the number of nodes and dimensions for 'routine_key' match the current working dataset, whether full or subsampled.
        if routine_key not in SSMcalcs_dic:
            SSMcalcs_dic[routine_key] = {k: ds.copy(deep=True) for k, ds in SSMcalcs_dic[pO2_subdir].items()}
        
        # Store results in separate dictionary keys (matches old workflow pattern)
        SSMcalcs_dic[routine_key][key][f'Mindex_{taxa}_routine'] = routine_mi_xarray
        
        print(f"    ✓ Routine metabolic index calculations complete for {key}")
        print(f"    Results summary (using {organism_weight_grams}g organisms):")
        print(f"      Routine MI range: {routine_mi_flat.min():.3f} to {routine_mi_flat.max():.3f}")
        print(f"      Routine stress fraction (MI<1): {(routine_mi_flat < 1).mean():.1%}")
        
        # Immediate cleanup after each calculation to save memory
        del routine_mi_flat, routine_mi_xarray

routine_calc_time = time.time() - routine_start_time
print(f"\n✓ Routine metabolism calculations complete for all datasets")
print(f"⏱️  Routine calculation time: {routine_calc_time:.2f} seconds")

#Export routine metabolism immediately after calculation 
print("\n" + "="*50)
print("EXPORTING: Routine Metabolism")
print("="*50)

# Start timing for first export block
routine_export_start_time = time.time()

# Export routine results (one at a time like old version)
for param_type in SSMinputsForMetabolic.keys():
    routine_key = f'{param_type}_3D_{taxa}_Mindex_routine'
    if routine_key in SSMcalcs_dic:
        print(f"Exporting {routine_key}...")
        export_dictionary_of_nc_datasets(
            dictionary_of_nc_datasets=SSMcalcs_dic[routine_key], 
            dictionary_name=routine_key, 
            output_dir=output_dir_export_nc, 
            encoding=encoding
        )
        print(f"✓ {routine_key} exported successfully")

routine_export_time = time.time() - routine_export_start_time
print(f"\n✓ Routine metabolism export complete")
print(f"⏱️  First export time: {routine_export_time:.2f} seconds")

##################################################################
#### Basal/SMR Metabolism Calculation ####
print("\n" + "="*50)
print("PROCESSING: Basal/SMR Metabolism")
print("="*50)

#Process using explicit mapping instead of searching
for param_type, config in SSMinputsForMetabolic.items():
    print(f"\nProcessing {param_type}...")
    
    # Direct reference to mapped subdirectories and variables
    pO2_subdir = config['pO2_subdir']
    pO2_var = config['pO2_var']
    temp_subdir = config['temp_subdir']
    temp_var = config['temp_var']
    
    # Process each dataset (e.g., 'exist', 'wqm_reference')
    for key in SSMcalcs_dic[pO2_subdir].keys():
        print(f"  Processing dataset: {key}")
        
        # Direct variable extraction using mapped names
        pO2_data = SSMcalcs_dic[pO2_subdir][key][pO2_var]
        temp_data = SSMcalcs_dic[temp_subdir][key][temp_var]
        
        # Flatten data for vectorized calculation
        pO2_flat = pO2_data.values.flatten()
        temp_flat = temp_data.values.flatten()
        weight_flat = np.full_like(pO2_flat, organism_weight_grams)
        
        print(f"    Calculating basal/SMR metabolic indices for {len(pO2_flat):,} data points...")
        
        # Process basal metabolism only
        basal_mi_flat = vectorized_mi_smr(pO2_flat, weight_flat, temp_flat)
        
        # Create xarray results
        basal_mi_xarray = pO2_data.copy(data=basal_mi_flat.reshape(pO2_data.shape))
        basal_mi_xarray.name = f'Mindex_{taxa}_basal'
        
        # Create separate dictionary keys for each output
        basal_key = f"{param_type}_3D_{taxa}_Mindex_basal"
        
        # Initialize dictionary using subsampled data template (matches current data dimensions)
        if basal_key not in SSMcalcs_dic:
            SSMcalcs_dic[basal_key] = {k: ds.copy(deep=True) for k, ds in SSMcalcs_dic[pO2_subdir].items()}
        
        # Store results in separate dictionary keys
        SSMcalcs_dic[basal_key][key][f'Mindex_{taxa}_basal'] = basal_mi_xarray
        
        print(f"    ✓ Basal/SMR metabolic index calculations complete for {key}")
        print(f"    Results summary (using {organism_weight_grams}g organisms):")
        print(f"      Basal/SMR MI range: {basal_mi_flat.min():.3f} to {basal_mi_flat.max():.3f}")
        print(f"      Basal/SMR stress fraction (MI<1): {(basal_mi_flat < 1).mean():.1%}")
        
        # Immediate cleanup after each calculation
        del basal_mi_flat, basal_mi_xarray

print(f"\n✓ Basal/SMR metabolism calculations complete for all datasets")

# Export basal metabolism immediately after calculation
print("\n" + "="*50)
print("EXPORTING: Basal/SMR Metabolism")
print("="*50)

# Export basal results (one at a time like old version)
for param_type in SSMinputsForMetabolic.keys():
    basal_key = f'{param_type}_3D_{taxa}_Mindex_basal'
    if basal_key in SSMcalcs_dic:
        print(f"Exporting {basal_key}...")
        export_dictionary_of_nc_datasets(
            dictionary_of_nc_datasets=SSMcalcs_dic[basal_key], 
            dictionary_name=basal_key, 
            output_dir=output_dir_export_nc, 
            encoding=encoding
        )
        print(f"✓ {basal_key} exported successfully")

print(f"\n✓ Basal/SMR metabolism export complete")

##################################################################
#### Routine CI Upper Calculation ####
print("\n" + "="*50)
print("PROCESSING: Routine CI Upper")
print("="*50)

# Process using explicit mapping instead of searching
for param_type, config in SSMinputsForMetabolic.items():
    print(f"\nProcessing {param_type}...")
    
    # Direct reference to mapped subdirectories and variables
    pO2_subdir = config['pO2_subdir']
    pO2_var = config['pO2_var']
    temp_subdir = config['temp_subdir']
    temp_var = config['temp_var']
    
    # Process each dataset (e.g., 'exist', 'wqm_reference')
    for key in SSMcalcs_dic[pO2_subdir].keys():
        print(f"  Processing dataset: {key}")
        
        # Direct variable extraction using mapped names
        pO2_data = SSMcalcs_dic[pO2_subdir][key][pO2_var]
        temp_data = SSMcalcs_dic[temp_subdir][key][temp_var]
        
        # Flatten data for vectorized calculation
        pO2_flat = pO2_data.values.flatten()
        temp_flat = temp_data.values.flatten()
        weight_flat = np.full_like(pO2_flat, organism_weight_grams)
        
        print(f"    Calculating routine CI upper for {len(pO2_flat):,} data points...")
        
        # Process routine CI upper only
        routine_upper_flat = vectorized_upper_routine(pO2_flat, weight_flat, temp_flat)
        
        # Create xarray results
        routine_upper_xarray = pO2_data.copy(data=routine_upper_flat.reshape(pO2_data.shape))
        routine_upper_xarray.name = f'Mindex_{taxa}_routine_ci_upper'
        
        # Create separate dictionary keys for each output
        routine_upper_key = f"{param_type}_3D_{taxa}_Mindex_routine_ci_upper"
        
        # Initialize dictionary using subsampled data template (matches current data dimensions)
        if routine_upper_key not in SSMcalcs_dic:
            SSMcalcs_dic[routine_upper_key] = {k: ds.copy(deep=True) for k, ds in SSMcalcs_dic[pO2_subdir].items()}
        
        # Store results in separate dictionary keys
        SSMcalcs_dic[routine_upper_key][key][f'Mindex_{taxa}_routine_ci_upper'] = routine_upper_xarray
        
        print(f"    ✓ Routine CI upper calculations complete for {key}")
        
        # Immediate cleanup after each calculation
        del routine_upper_flat, routine_upper_xarray

print(f"\n✓ Routine CI upper calculations complete for all datasets")

# Export routine CI upper immediately after calculation
print("\n" + "="*50)
print("EXPORTING: Routine CI Upper")
print("="*50)

# Export routine CI upper results (one at a time like old version)
for param_type in SSMinputsForMetabolic.keys():
    routine_upper_key = f'{param_type}_3D_{taxa}_Mindex_routine_ci_upper'
    if routine_upper_key in SSMcalcs_dic:
        print(f"Exporting {routine_upper_key}...")
        export_dictionary_of_nc_datasets(
            dictionary_of_nc_datasets=SSMcalcs_dic[routine_upper_key], 
            dictionary_name=routine_upper_key, 
            output_dir=output_dir_export_nc, 
            encoding=encoding
        )
        print(f"✓ {routine_upper_key} exported successfully")

print(f"\n✓ Routine CI upper export complete")

##################################################################
#### Routine CI Lower Calculation ####
print("\n" + "="*50)
print("PROCESSING: Routine CI Lower")
print("="*50)

# Process using explicit mapping instead of searching
for param_type, config in SSMinputsForMetabolic.items():
    print(f"\nProcessing {param_type}...")
    
    # Direct reference to mapped subdirectories and variables
    pO2_subdir = config['pO2_subdir']
    pO2_var = config['pO2_var']
    temp_subdir = config['temp_subdir']
    temp_var = config['temp_var']
    
    # Process each dataset (e.g., 'exist', 'wqm_reference')
    for key in SSMcalcs_dic[pO2_subdir].keys():
        print(f"  Processing dataset: {key}")
        
        # Direct variable extraction using mapped names
        pO2_data = SSMcalcs_dic[pO2_subdir][key][pO2_var]
        temp_data = SSMcalcs_dic[temp_subdir][key][temp_var]
        
        # Flatten data for vectorized calculation
        pO2_flat = pO2_data.values.flatten()
        temp_flat = temp_data.values.flatten()
        weight_flat = np.full_like(pO2_flat, organism_weight_grams)
        
        print(f"    Calculating routine CI lower for {len(pO2_flat):,} data points...")
        
        # Process routine CI lower only
        routine_lower_flat = vectorized_lower_routine(pO2_flat, weight_flat, temp_flat)
        
        # Create xarray results
        routine_lower_xarray = pO2_data.copy(data=routine_lower_flat.reshape(pO2_data.shape))
        routine_lower_xarray.name = f'Mindex_{taxa}_routine_ci_lower'
        
        # Create separate dictionary keys for each output
        routine_lower_key = f"{param_type}_3D_{taxa}_Mindex_routine_ci_lower"
        
        # Initialize dictionary using subsampled data template (matches current data dimensions)
        if routine_lower_key not in SSMcalcs_dic:
            SSMcalcs_dic[routine_lower_key] = {k: ds.copy(deep=True) for k, ds in SSMcalcs_dic[pO2_subdir].items()}
        
        # Store results in separate dictionary keys
        SSMcalcs_dic[routine_lower_key][key][f'Mindex_{taxa}_routine_ci_lower'] = routine_lower_xarray
        
        print(f"    ✓ Routine CI lower calculations complete for {key}")
        
        # Immediate cleanup after each calculation
        del routine_lower_flat, routine_lower_xarray

print(f"\n✓ Routine CI lower calculations complete for all datasets")

# Export routine CI lower immediately after calculation
print("\n" + "="*50)
print("EXPORTING: Routine CI Lower")
print("="*50)

# Export routine CI lower results (one at a time like old version)
for param_type in SSMinputsForMetabolic.keys():
    routine_lower_key = f'{param_type}_3D_{taxa}_Mindex_routine_ci_lower'
    if routine_lower_key in SSMcalcs_dic:
        print(f"Exporting {routine_lower_key}...")
        export_dictionary_of_nc_datasets(
            dictionary_of_nc_datasets=SSMcalcs_dic[routine_lower_key], 
            dictionary_name=routine_lower_key, 
            output_dir=output_dir_export_nc, 
            encoding=encoding
        )
        print(f"✓ {routine_lower_key} exported successfully")

print(f"\n✓ Routine CI lower export complete")

#### Basal CI Upper Calculation ####
print("\n" + "="*50)
print("PROCESSING: Basal CI Upper")
print("="*50)

# Process using explicit mapping instead of searching
for param_type, config in SSMinputsForMetabolic.items():
    print(f"\nProcessing {param_type}...")
    
    # Direct reference to mapped subdirectories and variables
    pO2_subdir = config['pO2_subdir']
    pO2_var = config['pO2_var']
    temp_subdir = config['temp_subdir']
    temp_var = config['temp_var']
    
    # Process each dataset (e.g., 'exist', 'wqm_reference')
    for key in SSMcalcs_dic[pO2_subdir].keys():
        print(f"  Processing dataset: {key}")
        
        # Direct variable extraction using mapped names
        pO2_data = SSMcalcs_dic[pO2_subdir][key][pO2_var]
        temp_data = SSMcalcs_dic[temp_subdir][key][temp_var]
        
        # Flatten data for vectorized calculation
        pO2_flat = pO2_data.values.flatten()
        temp_flat = temp_data.values.flatten()
        weight_flat = np.full_like(pO2_flat, organism_weight_grams)
        
        print(f"    Calculating basal CI upper for {len(pO2_flat):,} data points...")
        
        # Process basal CI upper only
        basal_upper_flat = vectorized_upper_smr(pO2_flat, weight_flat, temp_flat)
        
        # Create xarray results
        basal_upper_xarray = pO2_data.copy(data=basal_upper_flat.reshape(pO2_data.shape))
        basal_upper_xarray.name = f'Mindex_{taxa}_basal_ci_upper'
        
        # Create separate dictionary keys for each output
        basal_upper_key = f"{param_type}_3D_{taxa}_Mindex_basal_ci_upper"
        
        # Initialize dictionary using subsampled data template (matches current data dimensions)
        if basal_upper_key not in SSMcalcs_dic:
            SSMcalcs_dic[basal_upper_key] = {k: ds.copy(deep=True) for k, ds in SSMcalcs_dic[pO2_subdir].items()}
        
        # Store results in separate dictionary keys
        SSMcalcs_dic[basal_upper_key][key][f'Mindex_{taxa}_basal_ci_upper'] = basal_upper_xarray
        
        print(f"    ✓ Basal CI upper calculations complete for {key}")
        
        # Immediate cleanup after each calculation
        del basal_upper_flat, basal_upper_xarray

print(f"\n✓ Basal CI upper calculations complete for all datasets")

# Export basal CI upper immediately after calculation
print("\n" + "="*50)
print("EXPORTING: Basal CI Upper")
print("="*50)

# Export basal CI upper results (one at a time like old version)
for param_type in SSMinputsForMetabolic.keys():
    basal_upper_key = f'{param_type}_3D_{taxa}_Mindex_basal_ci_upper'
    if basal_upper_key in SSMcalcs_dic:
        print(f"Exporting {basal_upper_key}...")
        export_dictionary_of_nc_datasets(
            dictionary_of_nc_datasets=SSMcalcs_dic[basal_upper_key], 
            dictionary_name=basal_upper_key, 
            output_dir=output_dir_export_nc, 
            encoding=encoding
        )
        print(f"✓ {basal_upper_key} exported successfully")

print(f"\n✓ Basal CI upper export complete")

#### Basal CI Lower Calculation ####
print("\n" + "="*50)
print("PROCESSING: Basal CI Lower")
print("="*50)

# Process using explicit mapping instead of searching
for param_type, config in SSMinputsForMetabolic.items():
    print(f"\nProcessing {param_type}...")
    
    # Direct reference to mapped subdirectories and variables
    pO2_subdir = config['pO2_subdir']
    pO2_var = config['pO2_var']
    temp_subdir = config['temp_subdir']
    temp_var = config['temp_var']
    
    # Process each dataset (e.g., 'exist', 'wqm_reference')
    for key in SSMcalcs_dic[pO2_subdir].keys():
        print(f"  Processing dataset: {key}")
        
        # Direct variable extraction using mapped names
        pO2_data = SSMcalcs_dic[pO2_subdir][key][pO2_var]
        temp_data = SSMcalcs_dic[temp_subdir][key][temp_var]
        
        # Flatten data for vectorized calculation
        pO2_flat = pO2_data.values.flatten()
        temp_flat = temp_data.values.flatten()
        weight_flat = np.full_like(pO2_flat, organism_weight_grams)
        
        print(f"    Calculating basal CI lower for {len(pO2_flat):,} data points...")
        
        # Process basal CI lower only
        basal_lower_flat = vectorized_lower_smr(pO2_flat, weight_flat, temp_flat)
        
        # Create xarray results
        basal_lower_xarray = pO2_data.copy(data=basal_lower_flat.reshape(pO2_data.shape))
        basal_lower_xarray.name = f'Mindex_{taxa}_basal_ci_lower'
        
        # Create separate dictionary keys for each output
        basal_lower_key = f"{param_type}_3D_{taxa}_Mindex_basal_ci_lower"
        
        # Initialize dictionary using subsampled data template (matches current data dimensions)
        if basal_lower_key not in SSMcalcs_dic:
            SSMcalcs_dic[basal_lower_key] = {k: ds.copy(deep=True) for k, ds in SSMcalcs_dic[pO2_subdir].items()}
        
        # Store results in separate dictionary keys
        SSMcalcs_dic[basal_lower_key][key][f'Mindex_{taxa}_basal_ci_lower'] = basal_lower_xarray
        
        print(f"    ✓ Basal CI lower calculations complete for {key}")
        
        # Immediate cleanup after each calculation
        del basal_lower_flat, basal_lower_xarray

print(f"\n✓ Basal CI lower calculations complete for all datasets")

# Export basal CI lower immediately after calculation
print("\n" + "="*50)
print("EXPORTING: Basal CI Lower")
print("="*50)

# Export basal CI lower results (one at a time like old version)
for param_type in SSMinputsForMetabolic.keys():
    basal_lower_key = f'{param_type}_3D_{taxa}_Mindex_basal_ci_lower'
    if basal_lower_key in SSMcalcs_dic:
        print(f"Exporting {basal_lower_key}...")
        export_dictionary_of_nc_datasets(
            dictionary_of_nc_datasets=SSMcalcs_dic[basal_lower_key], 
            dictionary_name=basal_lower_key, 
            output_dir=output_dir_export_nc, 
            encoding=encoding
        )
        print(f"✓ {basal_lower_key} exported successfully")

print(f"\n✓ Basal CI lower export complete")


##################################################################
# Calculate and display final timing information
overall_end_time = time.time()
total_processing_time = overall_end_time - overall_start_time

# Final cleanup matching old workflow
del output_dir_export_nc, encoding

print(f"\n" + "="*60)
print("✓ ALL METABOLIC INDEX CALCULATIONS AND EXPORTS COMPLETE")
print(f"✓ Results stored in {taxa}-specific dictionary keys")
print(f"✓ Memory-optimized: Each output type processed separately")
print(f"✓ All outputs exported to ../../../../SSM_output/SSM_metabolic")
print(f"✓ Ready for further processing")
print("="*60)

print(f"\n" + "="*60)
print("⏱️  TIMING SUMMARY")
print("="*60)
print(f"First calculation (routine) block time: {routine_calc_time:.2f} seconds") #routine calculation timing
print(f"First export (routine) block time: {routine_export_time:.2f} seconds") #routine export timing  
print(f"Total processing time: {total_processing_time:.2f} seconds") #total processing time
print(f"Species processed: {taxa}")
print(f"Data points per calculation: {len(pO2_flat):,}")
print("="*60)


 Input mapping configured for metabolic calculations:
  CalMinParam: pO2 from pO2_daily_min_kPa, temp from temp_daily_mean_CT
  CalMaxParam: pO2 from pO2_daily_max_kPa, temp from temp_daily_mean_CT

Processing subsampled SSM data through vectorized metabolic functions
Memory-optimized approach: Processing one output type at a time
Exporting to: ../../../../SSM_output//SSM_metabolic
Processing species: salmon
------------------------------------------------------------

PROCESSING: Routine Metabolism

Processing CalMinParam...
  Using pO2 from: CalMinParam_3D_pO2_daily_min_kPa[pO2_daily_min_kPa]
  Using temp from: CalMinParam_3D_temp_daily_mean_CT[temp_daily_mean_CT]
  Processing dataset: wqm_reference
    Data shapes: pO2=(361, 10, 3), temp=(361, 10, 3)
    Using organism weight: 6400g
    Calculating routine metabolic indices for 10,830 data points...
    ✓ Routine metabolic index calculations complete for wqm_reference
    Results summary (using 6400g organisms):
      Routine MI ra

## Depth averaging for 2d and filtering for excel timeseries for specific cells

In [20]:
def average_or_select_by_depth_dataset(param_dict):
    """
    Averages or selects the data by specific depth for each key in the dictionary containing Datasets making a copy that is effectively 2D vs original 3D.
    Parameters:
        param_dict (dict): Dictionary containing xarray Datasets.
    Returns:
        dict: Updated dictionary with averaged/selected data.
    """
    param_dict_2D = {key: dataset.copy(deep=True) for key, dataset in param_dict.items()}  # Create a deep copy for new  2D version

    for key in param_dict_2D:  # Process the deep copy to create the 2D version
        dataset = param_dict_2D[key]
        print(f"\n Starting on new dataset. Processing key '{key}' with dataset:")
        print(dataset, "\n")
        
        for var_name in dataset.data_vars:  # Each data array inside the dataset
            var_shape = dataset[var_name].shape
            print(f"Processing variable: {var_name} with shape {var_shape}")  # Debugging
            
            if var_shape[1] != 10:
                raise ValueError(f"Error: Variable '{var_name}' in dataset '{key}' has dim_1 (depth layers) size {var_shape[1]}, expected 10.")
            
            # Extract top layer (_tp)
            dataset[var_name + '_tp'] = dataset[var_name][:, 0, :]  

            # Extract bottom layer (_bt)
            dataset[var_name + '_bt'] = dataset[var_name][:, 9, :]  

            # Extract min across middle layers (_md)
            dataset[var_name + '_md'] = dataset[var_name][:, 1:9, :].min(dim='dim_1')

            # Extract average across middle layers (_mA)
            dataset[var_name + '_mA'] = dataset[var_name][:, 1:9, :].mean(dim='dim_1')

            # Extract Average across the whole water column (_wA)
            dataset[var_name + '_wA'] = dataset[var_name].mean(dim='dim_1')

            # Extract min across the whole water column (_wc)
            dataset[var_name + '_wc'] = dataset[var_name].min(dim='dim_1')

            print(f"Added all new data arrays with depth selection/averaging for {var_name}")

        # Remove original 3D variables
        for var_name in list(dataset.data_vars):  # Iterate through all data variables in the dataset
            if dataset[var_name].ndim == 3:  # Check if the variable has three dimensions
                print(f"Lastly, removing variable: {var_name}")  # Debugging: print the name of the variable being removed
                del dataset[var_name]  # Delete the original 3D variable
        
        param_dict_2D[key] = dataset  # Update the dictionary with the modified dataset

    return param_dict_2D  # Return the updated 2D dictionary

#shared inputs for call: 
output_dir_export_nc = f'{output_dir}/SSM_metabolic'  # Define the output directory for export
encoding = {'zlib': True, 'complevel': 4}  # Set compression level for export
#call:
for key in list(SSMcalcs_dic.keys()):  # Iterate over a list of keys from the dictionary
    if '3D' in key:  # Check if the key contains '3D'
        corresponding_2D_key = key.replace('3D', '2D')  # Replace '3D' with '2D' to create the new key
        SSMcalcs_dic[corresponding_2D_key] = average_or_select_by_depth_dataset(SSMcalcs_dic[key])  # Calculate the 2D dataset and assign it to the new key
        # Keep the original 3D data for further analysis and Excel export - don't delete for supercomputer workflow
        # del SSMcalcs_dic[key]  # REMOVED: Delete the original 3D key to free up memory
        # Export the 2D dataset
        export_dictionary_of_nc_datasets(
            dictionary_of_nc_datasets=SSMcalcs_dic[corresponding_2D_key],  # created 2d dataset to export
            dictionary_name=corresponding_2D_key,  # Name of the dataset
            output_dir=output_dir_export_nc,  # Output directory
            encoding=encoding  # Compression settings
        )

print(f"\nDepth averaging and 2D datasets exported\n")
print(f"✓ 3D data preserved for Excel export and further analysis")


 Starting on new dataset. Processing key 'wqm_reference' with dataset:
<xarray.Dataset> Size: 93kB
Dimensions:            (dim_0: 361, dim_1: 10, dim_2: 3)
Coordinates:
  * dim_0              (dim_0) datetime64[ns] 3kB 2014-01-05 ... 2014-12-31
    time               (dim_0) datetime64[ns] 3kB ...
    depth_fraction     (dim_1) float64 80B ...
    node_id            (dim_2) int64 24B ...
    node_id_OrigGis    (dim_2) int64 24B ...
    latitude_reproj    (dim_2) float64 24B ...
    longitude_reproj   (dim_2) float64 24B ...
    total_depth_m      (dim_2) float64 24B ...
Dimensions without coordinates: dim_1, dim_2
Data variables:
    pO2_daily_min_kPa  (dim_0, dim_1, dim_2) float64 87kB 19.37 19.48 ... 15.84 

Processing variable: pO2_daily_min_kPa with shape (361, 10, 3)
Added all new data arrays with depth selection/averaging for pO2_daily_min_kPa
Lastly, removing variable: pO2_daily_min_kPa

 Starting on new dataset. Processing key 'exist' with dataset:
<xarray.Dataset> Size: 93kB


In [21]:
#Filter the datasets in the dictionary by specific nodes.
import xarray as xr
import pandas as pd
import os

# Function to filter by specific nodes (see above to provide first specific nodes from GIS file)
def filter_by_specific_nodes(data_dict, specific_nodes):
    """
    Filter the datasets in the dictionary by specific nodes.

    Parameters:
    data_dict (dict): Dictionary containing xarray Datasets.
    specific_nodes (list): List of specific nodes to filter by (1-based indexing). i.e. row 1 from gis is adjusted to row 0 for the eqiv. in numpy dataset

    Returns:
    dict: New dictionary with filtered datasets.
    """
    specific_nodes_zero_based = [node - 1 for node in specific_nodes]  # Adjust to zero-based indexing
    new_data_dict = {}  # Initialize a new dictionary to store filtered datasets

    # Loop through the keys for datasets and each dataarray variable in the dictionary
    for dataset_name, dataset in data_dict.items():
        new_data_dict[dataset_name] = {}  # Initialize a new dictionary for each dataset
        for var_name, data_array in dataset.items():
            # Check dimensions to handle both 2D and 3D data correctly
            if data_array.ndim == 3:
                # 3D data: (time, depth, nodes) - filter dimension 2 (nodes)
                filtered_data = data_array[:, :, specific_nodes_zero_based]  # Filter by specific nodes
            elif data_array.ndim == 2:
                # 2D data: (time, nodes) - filter dimension 1 (nodes)
                filtered_data = data_array[:, specific_nodes_zero_based]  # Filter by specific nodes
            else:
                # Handle other cases or raise error
                raise ValueError(f"Unexpected data dimensions: {data_array.ndim} for variable {var_name}")
            
            new_data_dict[dataset_name][var_name] = filtered_data  # Save the filtered data

    return new_data_dict  # Return the new dictionary with filtered datasets


############################################################

# call: filter_by_specific_nodes function  
# IMPORTANT: Since data is already subsampled to 3 nodes [13789, 8841, 14409], 
# we need to use the NEW indices (0, 1, 2) not the original node numbers

print("Data is already subsampled to 3 nodes: [13789, 8841, 14409]")
print("In subsampled data:")
print("  Index 0 = original node 13789")
print("  Index 1 = original node 8841") 
print("  Index 2 = original node 14409")

# Select which subsampled nodes to export to Excel (using new 1-based indices)
specific_nodes = [1]  # Index 1 = original node 13789 (1-based indexing for the function)
# specific_nodes = [2]  # Index 2 = original node 8841 
# specific_nodes = [3]  # Index 3 = original node 14409
# specific_nodes = [1, 2, 3]  # All three subsampled nodes

print(f"Exporting subsampled node index: {specific_nodes} to Excel")

SSMcalcs_dic_excel_for_specific_nodes = {}# Initialize new dictionary

# Loop through each key in the original dictionary and call the function and save an filted version for specific nodes 
for key in SSMcalcs_dic.keys():  # Iterate over keys in SSM2014_dic
    new_key = key + "_SpecificNodes"  # Create new key name by appending "_SpecificNodes"
    SSMcalcs_dic_excel_for_specific_nodes[new_key] = filter_by_specific_nodes(SSMcalcs_dic[key], specific_nodes)  # Apply filter function and assign to new key

print(f"✓ Filtered data ready for Excel export")

Data is already subsampled to 3 nodes: [13789, 8841, 14409]
In subsampled data:
  Index 0 = original node 13789
  Index 1 = original node 8841
  Index 2 = original node 14409
Exporting subsampled node index: [1] to Excel
✓ Filtered data ready for Excel export


## DEBUG: Quality Assurance Validation

This section provides comprehensive QA testing to validate the metabolic index calculations against Tim Essington's original test cases. The validation includes:

1. **Original Function Testing**: Reproduce Tim's four reference test cases using his exact code
2. **Vectorized Function Validation**: Verify that vectorized functions produce identical results 
3. **Pipeline Integration Testing**: Confirm the entire SSM processing pipeline works correctly
4. **Biological Relationship Validation**: Check expected patterns (SMR > routine, temperature effects, etc.)

These tests ensure the production pipeline maintains scientific accuracy while providing the performance needed for large SSM datasets.

### Reference Test Cases from Tim Essington

Reproduce the exact test cases we applied to Tim Essington's original code to for validation. These four test cases demonstrate the expected behavior of the metabolic index function under controlled conditions:

1. **Test 1**: Baseline routine metabolism (pO2=V, should equal 1.0)
2. **Test 2**: Standard metabolic rate comparison (should be > 1.0) 
3. **Test 3**: Temperature effect (higher temp should decrease MI)
4. **Test 4**: Body size effect (larger organisms should have lower MI)

In [22]:
# =============================================================================
# QA ON ORIGINAL CODE FROM TIM ESSINGTON FOR TESTS 1-4
# =============================================================================
# This reproduces the exact output from Tim's original code for comparison

print("="*80)
print("DEBUG: QA ON ORIGINAL CODE FROM TIM ESSINGTON FOR TESTS 1-4:")
print("="*80)

# Use the exact same parameters as in Tim's original code
w = 5
temperature = 15
method = "routine"
# set up parameter array: based on Chinook Salmon
betas = np.array([ 1.58422927, -0.04328307,  0.17567401, -0.32428962 ])
# set up variance covariance matrix of parameter estimates (based on Chinook Salmon)
var_covar = np.array([
    [  0.173846857 , 1.326809e-02, -0.073963952,  6.287643e-03],
    [0.013268092,  8.014129e-03, -0.004246992,  1.119302e-05],
    [-0.073963952, -4.246992e-03,  0.035758738, -2.752385e-03],
    [0.006287643,  1.119302e-05, -0.002752385,  2.331340e-02]])

# Test 1: call function, with pO2 = V and method = "routine".  Should return 1
pO2 = np.exp(betas[ 0 ]) # set pO2 to V
result = calc_mi(pO2, w, temperature, betas, var_covar, method= "routine", confidence_level=0.95)
print("Test 1 - Routine metabolism (should = 1.0):")
print(f"MI: {result['mi']}")
print(f"Lower bound: {result['lower_bound']}")
print(f"Upper bound: {result['upper_bound']}")

# Test 2: repeat but method = "smr", mi should be >1
result = calc_mi(pO2, w, temperature, betas, var_covar, method= "smr", confidence_level=0.95)
print("\nTest 2 - SMR metabolism (should > 1.0):")
print(f"MI: {result['mi']}")
print(f"Lower bound: {result['lower_bound']}")
print(f"Upper bound: {result['upper_bound']}")

# Test 3: increase temperature, mi should go down
temperature = 20
result = calc_mi(pO2, w, temperature, betas, var_covar, method= "routine", confidence_level=0.95)
print("\nTest 3 - Higher temperature (should decrease MI):")
print(f"MI: {result['mi']}")
print(f"Lower bound: {result['lower_bound']}")
print(f"Upper bound: {result['upper_bound']}")

# Test 4: increase w, MI should go down
w = 4000
temperature = 15
result = calc_mi(pO2, w, temperature, betas, var_covar, method= "routine", confidence_level=0.95)
print("\nTest 4 - Larger body size (should decrease MI):")
print(f"MI: {result['mi']}")
print(f"Lower bound: {result['lower_bound']}")
print(f"Upper bound: {result['upper_bound']}")

print("\n" + "="*80)
print("END OF ORIGINAL TIM ESSINGTON CODE QA TESTS 1-4")
print("="*80)

DEBUG: QA ON ORIGINAL CODE FROM TIM ESSINGTON FOR TESTS 1-4:
Test 1 - Routine metabolism (should = 1.0):
MI: 1.0
Lower bound: 0.4416639693994982
Upper bound: 2.2641647706957735

Test 2 - SMR metabolism (should > 1.0):
MI: 1.3830478074262176
Lower bound: 0.5636549567839213
Upper bound: 3.3936031513686356

Test 3 - Higher temperature (should decrease MI):
MI: 0.8863271883375078
Lower bound: 0.49374434953562796
Upper bound: 1.5910579746889557

Test 4 - Larger body size (should decrease MI):
MI: 0.7487646847944787
Lower bound: 0.2330737289969287
Upper bound: 2.40545579979442

END OF ORIGINAL TIM ESSINGTON CODE QA TESTS 1-4
